In [ ]:

import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import multiprocessing as mp
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import scipy.io
from joblib import Parallel, delayed
import scipy.special as scipy_special
import scipy.stats as scipy_stats
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler



MAT_PATH = (
    r"C:\Users\Melissa\MATLAB\neuralAnalysis\Regression\manyPt"
    r"\10Pts_HPC_GPT251024_100PCs.mat"
)
OUTPUT_DIR = (
    r"C:\Users\Melissa\MATLAB\neuralAnalysis\Regression\manyPt\output"
)

X_KEY = "X"
Y_KEY = "rates"

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 5


ALPHAS = np.logspace(-3, 3, 20)

N_SHUFFLES = 100

SHUF_METHOD = "circular"      # "circular" or "block"
MIN_CIRCULAR_LAG = 100
BLOCK_SIZE = 100

BASE_SEED = 20260716
MAX_ITER = 500
TOL = 1e-5


N_JOBS = min(8, max(1, mp.cpu_count() - 1))
PARALLEL_BACKEND = "threading"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# GENERAL UTILITIES
# =============================================================================

def poisson_ll(y: np.ndarray, mu: np.ndarray) -> float:
    """Full Poisson log likelihood, including log(y!)."""
    y = np.asarray(y, dtype=float)
    mu = np.clip(np.asarray(mu, dtype=float), 1e-12, None)
    return float(np.sum(y * np.log(mu) - mu - scipy_special.gammaln(y + 1.0)))


def safe_pearsonr(y: np.ndarray, pred: np.ndarray) -> Tuple[float, float]:
    y = np.asarray(y)
    pred = np.asarray(pred)
    if (
        y.size < 2
        or np.std(y) == 0
        or np.std(pred) == 0
        or np.unique(y).size < 2
        or np.unique(pred).size < 2
    ):
        return np.nan, np.nan
    r, p = scipy_stats.pearsonr(y, pred)
    return float(r), float(p)


def poisson_deviance_per_observation(
    y: np.ndarray, mu: np.ndarray
) -> np.ndarray:
    """Exact Poisson deviance contribution for each observation."""
    y = np.asarray(y, dtype=float)
    mu = np.clip(np.asarray(mu, dtype=float), 1e-12, None)
    log_term = np.zeros_like(y, dtype=float)
    positive = y > 0
    log_term[positive] = y[positive] * np.log(y[positive] / mu[positive])
    return 2.0 * (log_term - (y - mu))


def contiguous_test_folds(
    indices: np.ndarray, n_splits: int
) -> List[np.ndarray]:
    """
    Divide the supplied ORIGINAL observation indices into contiguous validation
    blocks without first concatenating separated temporal segments.
    """
    indices = np.asarray(indices, dtype=int)
    if indices.size < n_splits:
        raise ValueError(
            f"Only {indices.size} observations are available for {n_splits} folds."
        )

    # Split each contiguous run separately, then assign resulting chunks to folds
    # in temporal order. This prevents a validation block from crossing an outer
    # test gap.
    run_breaks = np.where(np.diff(indices) != 1)[0] + 1
    runs = np.split(indices, run_breaks)

    chunks: List[np.ndarray] = []
    for run in runs:
        if run.size == 0:
            continue
        # Allocate approximately proportional chunks while keeping each chunk
        # contiguous in original time.
        n_run_chunks = max(
            1, int(round(n_splits * run.size / max(indices.size, 1)))
        )
        n_run_chunks = min(n_run_chunks, run.size)
        chunks.extend([c for c in np.array_split(run, n_run_chunks) if c.size])

    # Merge temporally adjacent chunks until exactly n_splits blocks remain.
    chunks = sorted(chunks, key=lambda x: int(x[0]))
    while len(chunks) > n_splits:
        lengths = [
            chunks[i].size + chunks[i + 1].size
            for i in range(len(chunks) - 1)
        ]
        j = int(np.argmin(lengths))
        chunks[j] = np.concatenate([chunks[j], chunks[j + 1]])
        del chunks[j + 1]

    while len(chunks) < n_splits:
        j = int(np.argmax([c.size for c in chunks]))
        if chunks[j].size < 2:
            raise ValueError("Cannot construct the requested number of inner folds.")
        left, right = np.array_split(chunks[j], 2)
        chunks[j:j + 1] = [left, right]

    return chunks


def outer_folds(n_obs: int, n_splits: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    all_idx = np.arange(n_obs, dtype=int)
    test_blocks = np.array_split(all_idx, n_splits)
    return [
        (np.setdiff1d(all_idx, test_idx, assume_unique=True), test_idx)
        for test_idx in test_blocks
    ]


def make_shuffle_index(n_obs: int, rng: np.random.Generator) -> np.ndarray:
    """Create one reproducible autocorrelation-preserving semantic shuffle."""
    if SHUF_METHOD.lower() == "circular":
        if n_obs <= 2 * MIN_CIRCULAR_LAG:
            raise ValueError(
                "Not enough observations for the requested minimum circular lag."
            )
        possible = np.arange(MIN_CIRCULAR_LAG, n_obs - MIN_CIRCULAR_LAG + 1)
        lag = int(rng.choice(possible))
        if bool(rng.integers(0, 2)):
            lag = -lag
        return np.roll(np.arange(n_obs, dtype=int), lag)

    if SHUF_METHOD.lower() == "block":
        if BLOCK_SIZE < 1:
            raise ValueError("BLOCK_SIZE must be at least 1.")
        blocks = [
            np.arange(start, min(start + BLOCK_SIZE, n_obs), dtype=int)
            for start in range(0, n_obs, BLOCK_SIZE)
        ]
        order = rng.permutation(len(blocks))
        return np.concatenate([blocks[i] for i in order])

    raise ValueError(
        f"Unknown SHUF_METHOD={SHUF_METHOD!r}; use 'circular' or 'block'."
    )


def calculate_edf(
    X_scaled: np.ndarray,
    mu_train: np.ndarray,
    alpha: float,
) -> float:
    """
    Effective degrees of freedom for sklearn PoissonRegressor.

    sklearn minimizes mean Poisson deviance plus alpha/2 * ||beta||^2, so the
    equivalent summed-likelihood penalty is alpha * n.
    """
    X_scaled = np.asarray(X_scaled, dtype=float)
    mu_train = np.clip(np.asarray(mu_train, dtype=float), 1e-12, None)
    n = X_scaled.shape[0]

    design = np.column_stack([np.ones(n), X_scaled])
    weighted = np.sqrt(mu_train)[:, None] * design
    xtwx = weighted.T @ weighted

    penalty = np.eye(design.shape[1]) * (alpha * n)
    penalty[0, 0] = 0.0

    system = xtwx + penalty
    try:
        influence = np.linalg.solve(system, xtwx)
    except np.linalg.LinAlgError:
        influence = np.linalg.pinv(system) @ xtwx
    return float(np.trace(influence))


def generalized_aic(
    y_train: np.ndarray,
    mu_train: np.ndarray,
    X_train_scaled: np.ndarray,
    alpha: float,
) -> Tuple[float, float]:
    """
    Generalized AIC for ridge Poisson regression:
        GAIC = -2 * unpenalized log-likelihood + 2 * effective df.
    """
    edf = calculate_edf(X_train_scaled, mu_train, alpha)
    gaic = -2.0 * poisson_ll(y_train, mu_train) + 2.0 * edf
    return float(gaic), float(edf)


def summarize_coefficients(
    rows: List[Dict[str, float]],
) -> pd.DataFrame:
    """Median ridge coefficient across outer folds for each predictor."""
    if not rows:
        return pd.DataFrame(
            columns=["neuron", "predictor_index", "coefficient"]
        )
    frame = pd.DataFrame(rows)
    return (
        frame.groupby(["neuron", "predictor_index"], as_index=False)
        .agg({"coefficient": "median"})
    )

# =============================================================================
# FEATURE CONSTRUCTION: DURATION AS AN ORDINARY PREDICTOR
# =============================================================================

@dataclass
class NonOffsetFeatureBuilder:
    """
    Fit centering values on training data only, then construct:
        centered PCs | centered duration | centered PC x centered duration

    A StandardScaler is subsequently fit on the constructed training features.
    """

    pc_mean_: Optional[np.ndarray] = None
    duration_mean_: Optional[float] = None

    def fit(self, X_base_train: np.ndarray) -> "NonOffsetFeatureBuilder":
        pcs = np.asarray(X_base_train[:, :-1], dtype=float)
        duration = np.asarray(X_base_train[:, -1], dtype=float)
        self.pc_mean_ = np.mean(pcs, axis=0)
        self.duration_mean_ = float(np.mean(duration))
        return self

    def transform(self, X_base: np.ndarray) -> np.ndarray:
        if self.pc_mean_ is None or self.duration_mean_ is None:
            raise RuntimeError("Feature builder has not been fitted.")
        pcs = np.asarray(X_base[:, :-1], dtype=float) - self.pc_mean_
        duration = (
            np.asarray(X_base[:, -1], dtype=float) - self.duration_mean_
        )[:, None]
        interactions = pcs * duration
        return np.hstack([pcs, duration, interactions])


def fit_preprocessor(
    X_base: np.ndarray, train_idx: np.ndarray
) -> Tuple[NonOffsetFeatureBuilder, StandardScaler]:
    builder = NonOffsetFeatureBuilder().fit(X_base[train_idx])
    train_features = builder.transform(X_base[train_idx])
    scaler = StandardScaler().fit(train_features)
    return builder, scaler


def transform_with_preprocessor(
    X_base: np.ndarray,
    indices: np.ndarray,
    builder: NonOffsetFeatureBuilder,
    scaler: StandardScaler,
) -> np.ndarray:
    return scaler.transform(builder.transform(X_base[indices]))


def select_alpha_nested(
    X_base: np.ndarray,
    y: np.ndarray,
    outer_train_idx: np.ndarray,
) -> float:
    """Select alpha by nested contiguous CV without rebuilding folds per alpha."""
    validation_blocks = contiguous_test_folds(outer_train_idx, N_SPLITS_INNER)


    prepared_folds = []
    for val_idx in validation_blocks:
        inner_train_idx = np.setdiff1d(
            outer_train_idx, val_idx, assume_unique=True
        )
        builder, scaler = fit_preprocessor(X_base, inner_train_idx)
        X_inner_train = transform_with_preprocessor(
            X_base, inner_train_idx, builder, scaler
        )
        X_inner_val = transform_with_preprocessor(
            X_base, val_idx, builder, scaler
        )
        prepared_folds.append(
            (inner_train_idx, val_idx, X_inner_train, X_inner_val)
        )

    best_alpha = None
    best_total_ll = -np.inf

    for alpha in ALPHAS:
        total_ll = 0.0
        for (
            inner_train_idx,
            val_idx,
            X_inner_train,
            X_inner_val,
        ) in prepared_folds:
            model = PoissonRegressor(
                alpha=float(alpha), max_iter=MAX_ITER, tol=TOL
            )
            model.fit(X_inner_train, y[inner_train_idx])
            mu_val = model.predict(X_inner_val)
            total_ll += poisson_ll(y[val_idx], mu_val)

        if total_ll > best_total_ll:
            best_total_ll = total_ll
            best_alpha = float(alpha)

    if best_alpha is None:
        raise RuntimeError("Alpha selection failed.")
    return best_alpha


def fit_outer_model(
    X_base: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    fixed_alpha: Optional[float] = None,
) -> Dict[str, object]:
    """Fit one outer-fold model, optionally using a supplied alpha."""
    best_alpha = (
        select_alpha_nested(X_base, y, train_idx)
        if fixed_alpha is None
        else float(fixed_alpha)
    )
    builder, scaler = fit_preprocessor(X_base, train_idx)

    X_train = transform_with_preprocessor(X_base, train_idx, builder, scaler)
    X_test = transform_with_preprocessor(X_base, test_idx, builder, scaler)

    model = PoissonRegressor(
        alpha=best_alpha, max_iter=MAX_ITER, tol=TOL
    ).fit(X_train, y[train_idx])

    mu_train = np.clip(model.predict(X_train), 1e-12, None)
    mu_test = np.clip(model.predict(X_test), 1e-12, None)

    aic, edf = generalized_aic(
        y[train_idx], mu_train, X_train, best_alpha
    )

    return {
        "best_alpha": best_alpha,
        "builder": builder,
        "scaler": scaler,
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "mu_train": mu_train,
        "mu_test": mu_test,
        "AIC": aic,
        "edf": edf,
        "AIC_pred": -2.0 * poisson_ll(y[test_idx], mu_test),
    }


def process_neuron(
    neuron_idx: int,
    X_original: np.ndarray,
    rates: np.ndarray,
    fold_list: Sequence[Tuple[np.ndarray, np.ndarray]],
    X_shuffled_list: Sequence[np.ndarray],
):
    print(f"Neuron {neuron_idx}")
    y = np.asarray(rates[:, neuron_idx], dtype=float)

    if np.all(y == 0) or np.std(y) == 0:
        return None

    n_obs = y.size
    deviance_vec = np.full(n_obs, np.nan, dtype=np.float32)
    coef_rows: List[Dict[str, float]] = []
    fold_rows: List[Dict[str, float]] = []

    ll_real_total = 0.0
    ll_null_total = 0.0
    ll_shuf_totals = np.zeros(N_SHUFFLES, dtype=float)

    # Store fold-specific shuffled outputs 
    shuf_fold_metrics: List[List[Dict[str, float]]] = [
        [] for _ in range(N_SHUFFLES)
    ]

    for fold_id, (train_idx, test_idx) in enumerate(fold_list):
        real = fit_outer_model(X_original, y, train_idx, test_idx)
        mu_test = real["mu_test"]
        ll_real = poisson_ll(y[test_idx], mu_test)

        mu0 = max(float(np.mean(y[train_idx])), 1e-12)
        mu_null_test = np.full(test_idx.size, mu0)
        ll_null = poisson_ll(y[test_idx], mu_null_test)

        ll_real_total += ll_real
        ll_null_total += ll_null
        deviance_vec[test_idx] = poisson_deviance_per_observation(
            y[test_idx], mu_test
        ).astype(np.float32)

        corr, corr_p = safe_pearsonr(y[test_idx], mu_test)

        # Save ridge coefficients 
        for predictor_index, coef in enumerate(real["model"].coef_):
            coef_rows.append(
                {
                    "neuron": neuron_idx,
                    "predictor_index": predictor_index,
                    "iteration": fold_id,
                    "coefficient": float(coef),
                }
            )

         # shuffled model
        for shuffle_id, X_shuf in enumerate(X_shuffled_list):
            shuffled = fit_outer_model(
                X_shuf,
                y,
                train_idx,
                test_idx,
                fixed_alpha=float(real["best_alpha"]),
            )

            mu_shuf_test = shuffled["mu_test"]
            ll_shuf = poisson_ll(y[test_idx], mu_shuf_test)
            ll_shuf_totals[shuffle_id] += ll_shuf
            corr_shuf, corr_shuf_p = safe_pearsonr(
                y[test_idx], mu_shuf_test
            )

            shuf_fold_metrics[shuffle_id].append(
                {
                    "ll": ll_shuf,
                    "edf": float(shuffled["edf"]),
                    "AIC": float(shuffled["AIC"]),
                    "AIC_pred": float(shuffled["AIC_pred"]),
                    "corr": corr_shuf,
                    "corr_pval": corr_shuf_p,
                    "best_alpha": float(shuffled["best_alpha"]),
                }
            )

        fold_rows.append(
            {
                "ll_real": ll_real,
                "edf": float(real["edf"]),
                "pseudo_r2": np.nan,  
                "AIC": float(real["AIC"]),
                "AIC_pred": float(real["AIC_pred"]),
                "corr": corr,
                "corr_pval": corr_p,
                "best_alpha": float(real["best_alpha"]),
            }
        )

    real_frame = pd.DataFrame(fold_rows)
    summary = real_frame.median(numeric_only=True).to_dict()

    pseudo_r2_cv = (
        1.0 - ll_real_total / ll_null_total
        if ll_null_total != 0
        else np.nan
    )

    # Average each output over all permutation-fold fits.
    shuf_flat = [
        metric
        for shuffle_metrics in shuf_fold_metrics
        for metric in shuffle_metrics
    ]
    shuf_frame = pd.DataFrame(shuf_flat)

    mean_shuf_total_ll = float(np.mean(ll_shuf_totals))
    pseudo_r2_shuf_cv = (
        1.0 - mean_shuf_total_ll / ll_null_total
        if ll_null_total != 0
        else np.nan
    )

    p_value = (
        1.0 + np.sum(ll_shuf_totals >= ll_real_total)
    ) / (N_SHUFFLES + 1.0)

    summary.update(
        {
            "neuron": neuron_idx,
            "ll_real": float(ll_real_total),
            "ll_xshuf": mean_shuf_total_ll,
            "edf_shuf": float(shuf_frame["edf"].mean()),
            "pseudo_r2": float(pseudo_r2_cv),
            "pseudo_r2_shuf": float(pseudo_r2_shuf_cv),
            "AIC_xshuf": float(shuf_frame["AIC"].mean()),
            "AIC_predShuf": float(shuf_frame["AIC_pred"].mean()),
            "corr_xshuf": float(shuf_frame["corr"].mean()),
            "corr_xshuf_pval": float(shuf_frame["corr_pval"].mean()),
            "ll_diff_xshuf": float(ll_real_total - mean_shuf_total_ll),
            "p_val_ll_diff_xshuf": float(p_value),
        }
    )

    return summary, deviance_vec, coef_rows, neuron_idx


def main() -> None:
    data = scipy.io.loadmat(MAT_PATH)
    X_original = np.asarray(data[X_KEY], dtype=float)
    rates = np.asarray(data[Y_KEY], dtype=float)

    if X_original.shape[0] != rates.shape[0]:
        raise ValueError("X and rates must contain the same number of rows.")
    if np.isnan(X_original).any() or np.isnan(rates).any():
        raise ValueError(
            "X or rates contains NaNs. Remove matching rows before running."
        )

    n_words, n_neurons = rates.shape
    fold_list = outer_folds(n_words, N_SPLITS_OUTER)


    shuffle_indices = [
        make_shuffle_index(
            n_words, np.random.default_rng(BASE_SEED + shuffle_id)
        )
        for shuffle_id in range(N_SHUFFLES)
    ]


    # The final column is word duration and remains unshuffled.
    X_shuffled_list = []
    for shuffle_idx in shuffle_indices:
        X_shuf = X_original.copy()
        X_shuf[:, :-1] = X_original[shuffle_idx, :-1]
        X_shuffled_list.append(X_shuf)

    fold_file = os.path.join(
        OUTPUT_DIR,
        f"fold_indices_{n_words}_{N_SPLITS_OUTER}.npy",
    )
    np.save(fold_file, np.array(fold_list, dtype=object))

    print(
        f"Running {n_neurons} neurons with N_JOBS={N_JOBS}, "
        f"backend={PARALLEL_BACKEND}"
    )
    assert PARALLEL_BACKEND == "threading"
    print("CONFIRMED: using ThreadingBackend; no process pickling will occur.")
    results = Parallel(
        n_jobs=N_JOBS,
        prefer="threads",
        require="sharedmem",
        verbose=10,
    )(
        delayed(process_neuron)(
            neuron_idx,
            X_original,
            rates,
            fold_list,
            X_shuffled_list,
        )
        for neuron_idx in range(n_neurons)
    )

    summary_records = []
    coef_rows = []
    deviance_columns = []
    kept_neurons = []

    for result in results:
        if result is None:
            continue
        summary, deviance, local_coef_rows, neuron_idx = result
        summary_records.append(summary)
        coef_rows.extend(local_coef_rows)
        deviance_columns.append(deviance)
        kept_neurons.append(neuron_idx)

    summary_df = pd.DataFrame(summary_records)

    if deviance_columns:
        deviance_df = pd.DataFrame(
            np.column_stack(deviance_columns),
            index=np.arange(n_words),
            columns=kept_neurons,
        )
    else:
        deviance_df = pd.DataFrame(index=np.arange(n_words))
    deviance_df.index.name = "word_index"

    coefficient_df = summarize_coefficients(coef_rows)

    summary_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "HPC10Pts_GPT25Pearson_100PCs_summary_per_neuron.csv",
        ),
        index=False,
    )
    deviance_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "HPC10Pts_GPT25Pearson_100PCs_deviance_metrics.csv",
        )
    )
    coefficient_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "HPC10Pts_GPT25Pearson_100PCs_coefficients_and_pvalues.csv",
        ),
        index=False,
    )


if __name__ == "__main__":
    main()
